# U2.L2 — FFJORD on 2D

**Session U2.L2** (lab, joint, asynchronous). Companion to the U2.L2 lecture notes, which carry
the reasoning; this notebook is the part you execute.

You build a continuous normalizing flow from scratch, train it under a frozen compute budget, and
record what the training does. The records are written to two directories,
`pain_evidence/two-moons/` and `pain_evidence/checkerboard/`, beside the file `pain_evidence/pair.json`.
**These exact files are read again in U3.L1.** Keep them where you can find them.

### How to read this notebook

Clone the public repository (<https://github.com/adricortes/flow-matching-labs>) and open this notebook from `labs/`:
the file alone does not have `common/` and `configs/`. Watch the proof clinic of U2.T2 first; the release message of the
week links it.

Cells marked **TODO** are yours. Everything else is given code: run it, read it, go on. Prompt cells ask for two to four
sentences of your own, in your own words. The solutions notebook is released by the instructor.

There is no live session. Parts 1 and 2 end with checkpoints that pass or say what is wrong. The notes give
reference readings for Parts 3, 4 and 6, each only after you have written your own answer; Part 5 has none. If a run fails, the
notes' section *If a run fails* says what to do, and you can ask in the thread for this week, in the channel that
the course uses for announcements.
Questions about this lab carry the tag `#lab`, questions about the proof clinic `#clinic`, and anything
that stops you `#blocked`, which is answered within a day.

**Time.** About 2.5 to 3.5 hours of your own work, plus training time that runs without you: two runs
of about 396 s (two-moons) and 365 s (checkerboard) on the reference computer (one CPU thread).

## Part 0 — Orientation and the week's map

### The package

| File | What it is |
|---|---|
| `labs/lab-u2l2.ipynb` | this notebook |
| `common/painplots.py` | the measurement helper, **given complete** (v1.1); its contract is frozen |
| `common/toy2d.py` | the datasets of U1.L1, unchanged |
| `configs/u2l2-budget.yaml` | the frozen budget: every setting of the training run |
| `configs/u2l2-rescue.yaml` | the reduced budget of rescue tier 2 |
| `common/weights/u2l2-*.pt` | one reference checkpoint per dataset (rescue tier 3) |
| `labs/u2l2/` | `README_week.md`, the reference evidence pair, the reference MLflow export |

### Settings, and one rule about hardware

The budget is calibrated on a CPU. A GPU is allowed, if you move the model and the data to it yourself. It is logged, and `cpu` against `gpu` is only a
description: two runs are comparable when they share the **pairing key**
`machine_id + pair_id + budget_version` (see `common/painplots.py`). The `pair_id` is minted below,
once, and stored in `pain_evidence/pair.json`.

### Where the numbers go

MLflow, local: `sqlite:///mlflow.db` next to this notebook, experiment `U2`. Tags `session=U2.L2`,
`dataset=<name>`, `hardware=<cpu|gpu>`, `machine_id`, `pair_id`. Curves are logged **after** each
run, in one batch.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import copy
import json
import warnings
import math
import random
import shutil
import statistics
import time

import matplotlib.pyplot as plt
import mlflow               # Part 7 needs it: a missing install stops here, not after the training
import numpy as np
import torch
import torch.nn as nn
import yaml
from IPython.display import Image, display
from torchdiffeq import odeint, odeint_adjoint

from common import painplots
from common.toy2d import EXTENT, checkerboard, density_heatmap, two_moons, warped_grid

torch.set_num_threads(1)    # one thread: the budget is calibrated on one
warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad")   # a harmless warning that prints a temp path

# --- the switches of this notebook --------------------------------------------------------
TIER = 0                    # rescue ladder: 0 none | 1 fallback learning rate | 2 reduced budget | 3 reference run.
                            # A tier applies to the training run that follows it (Part 2 or Part 5): set it, run this cell, then run the training cell of that dataset.
CONFIG = ROOT / "configs" / ("u2l2-rescue.yaml" if TIER == 2 else "u2l2-budget.yaml")

cfg = yaml.safe_load(CONFIG.read_text())
if TIER == 1:
    cfg["opt"]["lr"] = cfg["fallback_lr"]       # tier 1: the one change; it is recorded in the config, in its hash and in the MLflow parameters
GENERATORS = {"two-moons": two_moons, "checkerboard": checkerboard}
LOG2PI = math.log(2 * math.pi)
EVIDENCE = ROOT / "pain_evidence"
REFERENCE = ROOT / "labs" / "u2l2" / "reference"
DEVICE_KIND = "cpu"         # the code of this notebook runs on the CPU. Declare "gpu" only if you edit the given cells to
                            # run on a GPU yourself: the declaration is logged, nothing more.
DEVICE = "cuda" if DEVICE_KIND == "gpu" else "cpu"
RESULTS = globals().get("RESULTS", {})   # one entry per dataset; running this cell again keeps them

def set_all_seeds(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

PAIR_ID = painplots.mint_pair_id(EVIDENCE) if TIER != 3 else painplots.REFERENCE_PAIR_ID
print("torch", torch.__version__, "| budget", cfg["budget_version"], "| tier", TIER)
print("machine_id", painplots.machine_id(), "| pair_id", PAIR_ID, "| device", painplots.device_name(DEVICE))

### The rescue ladder

If a part of the training fails, use the first tier that applies. Each tier still produces valid evidence.

1. **The training stops with a non-finite loss** (or torchdiffeq stops with `underflow in dt`). Set `TIER = 1` in the cell above and run again: the
   learning rate becomes `fallback_lr` of the config. If it stops again, go to tier 2, and then to tier 3.
2. **One run needs more than 45 minutes on your computer** (the training prints the elapsed minutes every 500 iterations: stop it and use tier 2 as soon as it has run for 45 minutes). Set `TIER = 2`: the config becomes
   `configs/u2l2-rescue.yaml` (width 32, 1500 iterations, its own `budget_version`).
3. **The training is unusable.** Set `TIER = 3`: Parts 2 and 3 load the reference checkpoint and copy the
   shipped reference evidence (`evidence_origin=reference_pair`); your MLflow run carries `rescue=true`.
   The tolerance sweep of Part 4 runs in full on the checkpoint and is printed; the reference
   `pain_tolerance` files stay in the evidence directory, because their wall-clock belongs to the reference computer.

A tier applies to one dataset at a time: set `TIER`, run the settings cell, then run the training cell of that
dataset (Part 2 for two-moons, Part 5 for the checkerboard). Each run keeps its own settings, so two-moons at tier 0 with the checkerboard at tier 2 is allowed.
Tier 3 still needs Part 1 to pass; if Part 1 blocks you, tag the thread `#blocked`. The tag `rescue=true` of your MLflow run costs nothing in grading. It is for the instructor's statistics.

## Part 1 — Build the augmented system

Read the two boxes of the U2.T2 notes first (<https://adricortes.github.io/flow-matching-lectures/U2/U2-T2-notes.html>): *Instantaneous change of
variables* and *Hutchinson v2*. This notebook restates neither.

We integrate from $t = 1$ (data) to $t = 0$ (base) the state $[x;\ \Delta]$ with
$$\frac{\mathrm{d}}{\mathrm{d}t}\begin{bmatrix} x \\ \Delta \end{bmatrix} = \begin{bmatrix} u_t^\theta(x) \\ -\nabla\!\cdot u_t^\theta(x) \end{bmatrix},
\qquad \Delta(1) = 0, \qquad \log p_1(x_1) = \log p_0(x_0) - \Delta(0).$$
In the code the variable is named `delta_logp`, never `logp`, so that the sign is visible.
$p_0 = \mathcal{N}(0, I_2)$.

In [ ]:
class Field(nn.Module):
    """The learned field u_t^theta(x): R^2 -> R^2, an MLP with time concatenated (TODO 1.1).

    The solver calls it as f(t, x), time first (U0.T5), with t a scalar tensor. The
    network sees t as a (B,) column beside x: the course convention for time in a
    network signature. `depth` hidden layers of width `hidden`, tanh between them.
    """

    def __init__(self, hidden: int = 64, depth: int = 3, zero_head: bool = True) -> None:
        super().__init__()
        layers, width_in = [], 3
        for _ in range(depth):
            # TODO 1.1 (a): append a Linear layer from width_in to hidden and then a Tanh;
            # then set width_in to hidden.
            raise NotImplementedError
        layers.append(nn.Linear(hidden, 2))
        self.net = nn.Sequential(*layers)
        if zero_head:
            # The U0.T2 defaults, item 3: zero-init the last layer, so u = 0 at the start.
            nn.init.zeros_(self.net[-1].weight)
            nn.init.zeros_(self.net[-1].bias)

    def forward(self, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        # TODO 1.1 (b): t is a scalar tensor. Expand it to a (B,) column, put it beside x as a
        # third input column, and apply the network. The signature is f(t, x): time first.
        raise NotImplementedError

In [ ]:
# Checkpoint 1.1: the field
h = cfg["model"]["hidden"]
f = Field(h, cfg["model"]["depth"])
expected = (3 * h + h) + 2 * (h * h + h) + (h * 2 + 2)             # three hidden layers of width h
assert sum(p.numel() for p in f.parameters()) == expected, f"parameter count {sum(p.numel() for p in f.parameters())}, expected {expected}"
x = torch.randn(5, 2); t = torch.tensor(0.3)
assert f(t, x).shape == (5, 2), "u must be (B, 2)"
assert torch.count_nonzero(f(t, x)) == 0, "the head is zero-initialized (U0.T2 defaults, item 3): u = 0 at the start"
g = Field(h, cfg["model"]["depth"], zero_head=False)
assert not torch.allclose(g(torch.tensor(0.1), x), g(torch.tensor(0.9), x)), "the field does not depend on t"
assert float((g(t, 2 * x) - 2 * g(t, x) + g(t, 0 * x)).abs().max()) > 1e-4, "the field is affine in x: is there a Tanh between the layers?"
# the reference weights (width 64) must load into YOUR field and give the stored outputs, whatever the tier
fr = Field(64, 3)
fr.load_state_dict(torch.load(ROOT / "common" / "weights" / "u2l2-two-moons.pt", map_location="cpu")["state_dict"])
xr = torch.tensor([[0.5, -1.0], [1.5, 2.0]])
ur = torch.tensor([[0.6821611523628235, -1.448706865310669], [-2.5024664402008057, -0.6385168433189392]])
assert torch.allclose(fr(torch.tensor(0.37), xr), ur, atol=1e-5), "the reference weights give another output in your Field: check the activation, the order of the input columns and where t goes"
print(f"Checkpoint 1.1 passed: {expected} parameters, f(t, x) time first, zero head at init.")

In [ ]:
def exact_divergence(u: torch.Tensor, x: torch.Tensor, create_graph: bool) -> torch.Tensor:
    """div u = sum_i d u_i / d x_i, one autograd call per coordinate (TODO 1.2).

    In d = 2 this costs two backward passes and is the deterministic ground truth of
    every numerical check and every evaluation sweep in this lab.
    """
    # TODO 1.2: one torch.autograd.grad call per coordinate i, on u[:, i].sum(), with respect to x.
    # Keep column i of the result. Pass create_graph=create_graph and retain_graph=True. Sum over i.
    raise NotImplementedError

In [ ]:
# Checkpoint 1.2: the exact divergence on a field whose divergence you can write down
x = torch.tensor([[0.7, -1.2], [0.1, 2.0], [-0.4, 0.9]], requires_grad=True)
def u_of(x): return torch.stack([x[:, 0] * x[:, 1], x[:, 0] ** 2 + x[:, 1]], dim=1)    # div u = xi_2 + 1, with xi a point of the plane
assert torch.allclose(exact_divergence(u_of(x), x, False), x[:, 1].detach() + 1, atol=1e-6), "div u = xi_2 + 1"
assert exact_divergence(u_of(x), x, True).requires_grad, "with create_graph=True the divergence must stay differentiable"
assert not exact_divergence(u_of(x), x, False).requires_grad, "with create_graph=False the divergence keeps no graph: pass create_graph=create_graph (a kept graph changes no value and raises the wall-clock of Part 4)"
print("Checkpoint 1.2 passed: exact divergence, one autograd call per coordinate.")

### TODO 1.3 and 1.4 (a): the dynamics

The class below holds the right-hand side of the augmented system. Read its docstring first.
**The probe is drawn in `before_solve`, once per minibatch likelihood solve.** The solver calls
`forward` many times, and the adjoint calls it again while it recomputes the trajectory backward.
All of those calls use the same stored probe. Never draw a probe inside `forward`.

In [ ]:
class AugmentedDynamics(nn.Module):
    """d/dt [x; Delta] = [u_t^theta(x); -div u_t^theta(x)] (TODOs 1.3 and 1.4).

    The state is `[x (2), delta_logp (1)]`, plus two integral-loss slots when
    `regularize` is on (the RNODE penalties of Part 6, given). The solve runs from t = 1
    (data) to t = 0 (base) with delta_logp(1) = 0. Then, with Delta(t) = log p_t(x_t) - log p_1(x_1),
    delta_logp(0) = Delta(0) and log p_1(x_1) = log p_0(x_0) - delta_logp(0).

    HUTCHINSON, FIXED PROBE. `before_solve(x)` draws one Rademacher probe per example,
    ONCE per minibatch likelihood solve. The probe is stored and reused by every call of the
    solver and by every recomputation the adjoint performs. It is never drawn inside
    `forward`. `self.nfe` counts calls of the field (a vector-Jacobian product made inside a
    call is not another call).
    """

    def __init__(self, field: nn.Module, divergence: str = "hutchinson", regularize: bool = False,
                 weights: tuple[float, float] = (0.0, 0.0), probe_seed: int = 0) -> None:
        super().__init__()
        assert divergence in ("hutchinson", "exact")
        self.field, self.divergence, self.regularize, self.weights = field, divergence, regularize, weights
        self.nfe = 0
        self.probe = None
        self.audit = None            # a list of probe checksums, when the audit is on
        self.probe_gen = torch.Generator().manual_seed(probe_seed)

    def before_solve(self, x: torch.Tensor) -> None:
        """Sample the probe once for the coming likelihood solve (TODO 1.3)."""
        self.nfe = 0                                          # the count of calls restarts with every solve
        # TODO 1.3 (a): draw ONE Rademacher probe per example, the shape, dtype and device of x, values +1 or -1,
        # from self.probe_gen, and store it in self.probe. This method runs once per solve.
        raise NotImplementedError

    def forward(self, t: torch.Tensor, state: torch.Tensor) -> torch.Tensor:
        self.nfe += 1
        if self.audit is not None:
            self.audit.append(float((self.probe.double() * torch.arange(1, self.probe.numel() + 1,
                                     dtype=torch.float64).reshape(self.probe.shape)).sum()))
        create_graph = torch.is_grad_enabled()      # True in the adjoint pass and in direct backprop
        x = state[:, :2]
        with torch.enable_grad():
            if not x.requires_grad:
                x = x.detach().requires_grad_(True)     # forward solve: a fresh leaf
            # In the adjoint pass `state` already carries a graph: x must stay connected to it.
            u = self.field(t, x)
            if self.divergence == "exact":
                div = exact_divergence(u, x, create_graph)
                vjp = None
            else:
                # TODO 1.3 (b): the Hutchinson estimate of the divergence from ONE vector-Jacobian
                # product with the stored probe (never draw a probe here). Pass create_graph=create_graph.
                # vjp is v^T J, div is v^T J v.
                raise NotImplementedError
        # TODO 1.4 (a): the right-hand side of the augmented state [x; delta_logp]. The first
        # block is u. The second block is the derivative of delta_logp along the solve. Mind the sign.
        out = []                   # fill this list with the two blocks; the regularizer below appends to it
        raise NotImplementedError
        if self.regularize:
            if vjp is None:
                raise ValueError("the regularizer reuses the probe's product: use the Hutchinson mode")
            out += [-(u ** 2).sum(1, keepdim=True), -(vjp ** 2).sum(1, keepdim=True)]
        return torch.cat(out, dim=1)

The next function is given. `solve_augmented` integrates the state from $t = 1$ to $t = 0$ with
$\Delta(1) = 0$. It does not draw a probe: `before_solve` does.

In [ ]:
def solve_augmented(dyn: AugmentedDynamics, x1: torch.Tensor, *, rtol: float, atol: float,
                    method: str = "dopri5", adjoint: bool = True, **options) -> torch.Tensor:
    """Integrate the augmented state from t = 1 to t = 0; returns the state at t = 0 (given).

    The probe must already be set (`before_solve`). `adjoint=True` is the course default
    for training; `adjoint=False` differentiates through the solver steps instead.
    """
    extra = 2 if dyn.regularize else 0
    state1 = torch.cat([x1, x1.new_zeros(x1.shape[0], 1 + extra)], dim=1)
    t_span = x1.new_tensor([1.0, 0.0])
    solve = odeint_adjoint if adjoint else odeint
    return solve(dyn, state1, t_span, rtol=rtol, atol=atol, method=method, **options)[-1]

### The divergence-check ritual

Three witnesses on a field whose last layer is **not** zero. (With the zero head of the U0.T2 defaults the field is
$u = 0$, every divergence is $0$, and every check would pass from nothing.) (i) the exact autograd trace;
(ii) an independent trace from coordinate central differences, in float64; (iii) the Hutchinson sample
mean over a bank of probes, with its standard error. The two deterministic witnesses must agree to
numerical tolerance. The stochastic one is judged statistically: the exact value lies inside a band of
four standard errors.

In [ ]:
@torch.no_grad()
def central_difference_trace(field: nn.Module, t: torch.Tensor, x: torch.Tensor, eps: float = 1e-6):
    """Independent witness: coordinate central differences of u, in float64."""
    tr = torch.zeros(x.shape[0], dtype=x.dtype)
    for i in range(x.shape[1]):
        e = torch.zeros_like(x)
        e[:, i] = eps
        tr += (field(t, x + e)[:, i] - field(t, x - e)[:, i]) / (2 * eps)
    return tr


# Checkpoint 1.3: the divergence ritual
torch.set_default_dtype(torch.float64)
try:
    set_all_seeds(2026)
    fld = Field(64, 3, zero_head=False).double()
    x = two_moons(64, 3).double(); t = torch.tensor(0.5, dtype=torch.float64)
    xg = x.clone().requires_grad_(True)
    exact = exact_divergence(fld(t, xg), xg, False).detach()
    central = central_difference_trace(fld, t, x)
    assert float((exact - central).abs().max()) < 1e-7, "exact autograd trace and central-difference trace disagree"
    assert float(exact.abs().max()) > 1e-2, "this field has a trivial divergence: the check is vacuous"
    # the right-hand side you wrote, in exact mode: its second block is -div u
    dyn_exact = AugmentedDynamics(fld, "exact"); dyn_exact.before_solve(x)
    out = dyn_exact(t, torch.cat([x, x.new_zeros(len(x), 1)], dim=1))
    assert torch.allclose(out[:, :2], fld(t, x)), "the first block of the right-hand side is u"
    assert torch.allclose(-out[:, 2], exact, atol=1e-9), "the second block of the right-hand side is -div u"
    dyn = AugmentedDynamics(fld, "hutchinson", probe_seed=7)
    # your Hutchinson right-hand side against v^T J v with the stored probe v and J from autograd
    dyn.before_solve(x)
    assert dyn.probe.shape == x.shape and dyn.probe.dtype == x.dtype, "the probe has the shape and the dtype of x (TODO 1.3 a)"
    out = dyn(t, torch.cat([x, x.new_zeros(len(x), 1)], dim=1))
    for i in range(6):
        J = torch.autograd.functional.jacobian(lambda z: fld(t, z[None])[0], x[i])
        vJv = float(dyn.probe[i] @ J @ dyn.probe[i])
        assert abs(float(-out[i, 2]) - vJv) < 1e-9, "the second block is not the Hutchinson product v^T J v of the stored probe"
    # the regularizer's two extra columns reuse the same product: -|u|^2 and -|v^T J|^2
    dyn_r = AugmentedDynamics(fld, "hutchinson", regularize=True, weights=(0.01, 0.01), probe_seed=7); dyn_r.before_solve(x)
    out_r = dyn_r(t, torch.cat([x, x.new_zeros(len(x), 3)], dim=1))
    xg2 = x.clone().requires_grad_(True); u2 = fld(t, xg2); vjp2 = torch.autograd.grad(u2, xg2, dyn_r.probe)[0]
    assert out_r.shape[1] == 5 and torch.allclose(out_r[:, 4], -(vjp2 ** 2).sum(1), atol=1e-9), "the Hutchinson branch must leave the product in a variable named vjp (the regularizer reuses it)"
    # the probe comes from self.probe_gen: the same probe_seed gives the same probe whatever the global generator did
    d1 = AugmentedDynamics(fld, "hutchinson", probe_seed=7); torch.manual_seed(0); d1.before_solve(x)
    d2 = AugmentedDynamics(fld, "hutchinson", probe_seed=7); torch.manual_seed(1); d2.before_solve(x)
    d3 = AugmentedDynamics(fld, "hutchinson", probe_seed=8); d3.before_solve(x)
    assert torch.equal(d1.probe, d2.probe), "the probe depends on the global random generator: draw it from self.probe_gen"
    assert not torch.equal(d1.probe, d3.probe), "two probe seeds gave the same probe"
    # one call evaluates the field once, makes ONE product, and keeps its graph only when the solver differentiates it.
    # A second evaluation, a second product or a kept graph changes no value above and raises the wall-clock of your
    # evidence, so the calls are counted here.
    state = torch.cat([x, x.new_zeros(len(x), 1)], dim=1); made = []; evals = []; real_grad = torch.autograd.grad
    def counting_grad(*a, **k):
        kept = bool(k.get("create_graph", a[4] if len(a) > 4 else False))
        batched = k.get("is_grads_batched", a[7] if len(a) > 7 else False)      # a batched call is many products
        made.extend([kept] * (2 if batched else 1)); return real_grad(*a, **k)
    hook = fld.register_forward_hook(lambda *_: evals.append(1))
    torch.autograd.grad = counting_grad
    try:
        with torch.no_grad():
            d1(t, state)
        primal = made[:]; made.clear()
        d1(t, state)
    finally:
        torch.autograd.grad = real_grad; hook.remove()
    assert len(evals) == 2, "one call of the right-hand side evaluates the field once: make the product from the u computed above it (TODO 1.3 b). A second evaluation changes no value and raises the wall-clock of your evidence"
    if primal:
        assert len(primal) == 1 and len(made) == 1, "one call of the Hutchinson right-hand side makes ONE vector-Jacobian product, not a full Jacobian (TODO 1.3 b)"
        assert not primal[0], "under torch.no_grad() the product keeps no graph: pass create_graph=create_graph (TODO 1.3 b)"
    else:
        print("note: your product does not go through torch.autograd.grad, so this checkpoint could not count it. Use torch.autograd.grad: the count protects the wall-clock of your evidence.")
    K = 20000
    print(f"{'point':>5} {'exact (autograd)':>18} {'central diff':>14} {'Hutchinson mean':>16} {'std. error':>11}")
    for p in range(4):
        xp = x[p:p + 1].repeat(K, 1)
        dyn.before_solve(xp)                                      # one probe per row: a bank of K probes
        out = dyn(t, torch.cat([xp, xp.new_zeros(K, 1)], dim=1))  # your Hutchinson right-hand side, one call
        est = -out[:, 2].detach()                                 # its second block is minus the estimate
        mean, se = float(est.mean()), float(est.std() / math.sqrt(K))
        print(f"{p:>5} {float(exact[p]):>18.6f} {float(central[p]):>14.6f} {mean:>16.6f} {se:>11.6f}")
        assert se > 0 and abs(mean - float(exact[p])) <= 4 * se, "the exact value is outside four standard errors (or the estimate does not vary with the probe)"
        assert set(dyn.probe.unique().tolist()) == {-1.0, 1.0}, "a Rademacher probe takes the values +1 and -1"
finally:
    torch.set_default_dtype(torch.float32)
print("Checkpoint 1.3 passed: two deterministic witnesses agree; the stochastic estimate is inside its band.")

In [ ]:
def neg_log_likelihood(dyn: AugmentedDynamics, x1: torch.Tensor, *, rtol: float, atol: float,
                       adjoint: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
    """-log p_1(x_1) per example, and the final state. One fresh probe per call."""
    dyn.before_solve(x1)
    state0 = solve_augmented(dyn, x1, rtol=rtol, atol=atol, adjoint=adjoint)
    x0, delta_logp = state0[:, :2], state0[:, 2]
    # TODO 1.4 (b): log p_0(x_0) for p_0 = N(0, I_2), then log p_1(x_1) from the contract of the
    # notes (the variable stays named delta_logp, never logp). Return -log p_1 and state0.
    raise NotImplementedError

`evaluate_nll` is given. It computes the test NLL with the exact divergence on one batch, **through your
`neg_log_likelihood`**; the checks below and Part 4 use it.

In [ ]:
@torch.no_grad()
def evaluate_nll(field: nn.Module, x: torch.Tensor, *, rtol: float, atol: float) -> dict:
    """Test NLL with the EXACT divergence on one batch: isolates solver error from probe noise.

    It goes through `neg_log_likelihood`, so it uses your sign contract. The number of calls of the field
    is read from the dynamics after the solve."""
    dyn = AugmentedDynamics(field, divergence="exact")
    t0 = time.perf_counter()
    nll, _ = neg_log_likelihood(dyn, x, rtol=rtol, atol=atol, adjoint=False)
    wall = time.perf_counter() - t0
    return {"nll": float(nll.mean()), "nfe": dyn.nfe, "wall_s": wall}

The cell below is given. It defines the gradient check of PS2.5 (c): the **adjoint** gradient of the NLL, the
gradient by **direct** autograd through the solver steps, and a central finite difference of the same program,
all along one fixed direction in parameter space, in float64, with the probe fixed. It also defines
`BrokenDynamics`, a dynamics that differs from yours in one argument of one line.

In [ ]:
RITUAL_HIDDEN = 64
PROBE_FIXED = None

class BrokenDynamics(AugmentedDynamics):
    """The planted defect: the divergence's vector-Jacobian product is built without a graph."""

    def forward(self, t, state):
        self.nfe += 1
        x = state[:, :2]
        with torch.enable_grad():
            if not x.requires_grad:
                x = x.detach().requires_grad_(True)
            u = self.field(t, x)
            # Differs from AugmentedDynamics in one argument of one line (see the table of the checkpoint).
            vjp = torch.autograd.grad(u, x, self.probe, retain_graph=True)[0]
            div = (vjp * self.probe).sum(1)
        return torch.cat([u, -div[:, None]], dim=1)


def _loss_for(dyn_cls, field, x1, n_steps, adjoint):
    dyn = dyn_cls(field, "hutchinson")
    dyn.probe = PROBE_FIXED
    s0 = solve_augmented(dyn, x1, rtol=1e-6, atol=1e-8, method="rk4", adjoint=adjoint,
                         options={"step_size": 1.0 / n_steps})
    logp0 = -0.5 * (s0[:, :2] ** 2).sum(1) - LOG2PI
    return (-(logp0 - s0[:, 2])).mean()


def gradient_check() -> dict:
    """PS2.5(c): adjoint vs direct autograd vs central difference, probe fixed, float64."""
    global PROBE_FIXED
    set_all_seeds(2026)
    x1 = two_moons(32, 4).double()
    gen = torch.Generator().manual_seed(99)
    PROBE_FIXED = (torch.randint(0, 2, tuple(x1.shape), generator=gen) * 2 - 1).double()
    base = Field(RITUAL_HIDDEN, 3, zero_head=False).double()
    params = list(base.parameters())
    dgen = torch.Generator().manual_seed(2026)
    v = [torch.randn(p.shape, generator=dgen, dtype=torch.float64) for p in params]
    norm = math.sqrt(sum(float((vi ** 2).sum()) for vi in v))
    v = [vi / norm for vi in v]
    rows = []

    def directional(dyn_cls, n_steps, adjoint):
        f = copy.deepcopy(base)
        loss = _loss_for(dyn_cls, f, x1, n_steps, adjoint)
        grads = torch.autograd.grad(loss, list(f.parameters()), allow_unused=True)
        return sum(float((g * vi).sum()) for g, vi in zip(grads, v) if g is not None), float(loss)

    def central(n_steps, eps=1e-6):
        vals = []
        for sign in (+1, -1):
            f = copy.deepcopy(base)
            with torch.no_grad():
                for p, vi in zip(f.parameters(), v):
                    p.add_(sign * eps * vi)
            vals.append(float(_loss_for(AugmentedDynamics, f, x1, n_steps, False)))
        return (vals[0] - vals[1]) / (2 * eps)

    for n_steps in (8, 32):
        fd = central(n_steps)
        d_direct, loss = directional(AugmentedDynamics, n_steps, False)
        d_adj, _ = directional(AugmentedDynamics, n_steps, True)
        d_broken, _ = directional(BrokenDynamics, n_steps, False)
        rel = lambda a: abs(a - fd) / abs(fd)  # noqa: E731
        rows.append({"n_steps": n_steps, "loss": loss, "fd": fd, "direct": d_direct, "adjoint": d_adj,
                     "broken_direct": d_broken, "rel_direct": rel(d_direct), "rel_adjoint": rel(d_adj),
                     "rel_broken": rel(d_broken)})
    return {"hidden": RITUAL_HIDDEN, "batch": 32, "rows": rows, "fd_eps": 1e-6, "scheme": "torchdiffeq rk4 (3/8 rule)"}

In [ ]:
# Checkpoint 1.4: five checks. (a) the untrained-baseline check, (b) the closed-form sign check, (c) the probe audit, (d) the gradient check, (e) the gradient of your NLL.
torch.set_default_dtype(torch.float64)
try:
    # (a) the untrained model (zero head) is u = 0, so the NLL is known in closed form (fixed seed, exact divergence)
    for name, gen in GENERATORS.items():
        xs = gen(512, 1).double()
        r = evaluate_nll(Field(64, 3, zero_head=True).double(), xs, rtol=1e-8, atol=1e-10)
        closed = float((0.5 * (xs ** 2).sum(1)).mean() + LOG2PI)
        assert abs(r["nll"] - closed) < 1e-9, f"{name}: untrained NLL {r['nll']:.9f}, closed form {closed:.9f}"
    print("(a) untrained model: NLL equals the closed form.")

    # (b) a field with a known answer: u = c x. Then log p_1 is the log density of N(0, e^(2c) I).
    c = 0.5
    class Linear(nn.Module):
        def forward(self, t, x): return c * x
    xs = two_moons(512, 1).double()
    nll_mine, _ = neg_log_likelihood(AugmentedDynamics(Linear(), "exact"), xs, rtol=1e-10, atol=1e-12, adjoint=False)
    closed = -0.5 * (xs ** 2).sum(1) * math.exp(-2 * c) - LOG2PI - 2 * c
    err = float((-nll_mine - closed).abs().max())       # YOUR neg_log_likelihood, against the closed form
    assert err < 1e-8, f"u = c x: log p_1 is off by {err:.2e}; an error of 2dc = {2*2*c} means the wrong sign of delta_logp"
    print(f"(b) u = {c} x: closed form reproduced to {err:.1e}.")

    # (c) audit of the fixed probe: one checksum per solve, across the forward and the adjoint calls
    set_all_seeds(5)
    dyn = AugmentedDynamics(Field(16, 3, zero_head=False).double(), "hutchinson", probe_seed=11)
    dyn.audit = []; x1 = two_moons(32, 4).double()
    nll, st0 = neg_log_likelihood(dyn, x1, rtol=1e-6, atol=1e-8)
    assert st0.shape == (len(x1), 3), "neg_log_likelihood returns (-log p_1, the final state of shape (B, 3))"
    n_fwd = len(dyn.audit); assert dyn.nfe == n_fwd, "dyn.nfe must count the calls of ONE solve: reset it in before_solve"
    assert nll.requires_grad, "neg_log_likelihood returned a tensor with no graph: look for .detach(), .item() or a numpy call in TODO 1.4 (b)"
    try:
        nll.mean().backward()
    except RuntimeError as err:
        raise AssertionError(f"the backward pass failed ({str(err)[:80]}...): the product of TODO 1.3 (b) must pass create_graph=create_graph and must not pass retain_graph=False") from err
    n_all = len(dyn.audit)
    first = set(dyn.audit); neg_log_likelihood(dyn, x1, rtol=1e-6, atol=1e-8); second = set(dyn.audit[n_all:])
    assert len(first) == 1 and len(second) == 1 and first != second, "the probe changed inside a solve, or did not change between solves"
    print(f"(c) probe audit: the stored probe had one value in {n_fwd} primal and {n_all - n_fwd} adjoint calls; the next solve stored a new one.")

    # (d) the gradient-check table (PS2.5 c): the probe is fixed, so the loss is a deterministic function of theta
    g = gradient_check()
    print(f"(d) gradient check, float64, directional derivative along one fixed unit direction in parameter space:")
    print(f"{'RK4 steps':>9} {'central diff':>14} {'direct':>14} {'adjoint':>14} {'rel direct':>11} {'rel adjoint':>12} {'rel broken':>11}")
    for r in g["rows"]:
        print(f"{r['n_steps']:>9} {r['fd']:>14.9f} {r['direct']:>14.9f} {r['adjoint']:>14.9f} "
              f"{r['rel_direct']:>11.1e} {r['rel_adjoint']:>12.1e} {r['rel_broken']:>11.1e}")
        assert r["rel_direct"] < 1e-6 and r["rel_adjoint"] < 1e-6, "direct or adjoint gradient is outside the stated magnitude 1e-6: compare with the last column (BrokenDynamics): does the product of TODO 1.3 (b) keep its graph?"
        assert r["rel_broken"] > 1e-2, "the planted defect should fail this comparison"

    # (e) the gradient of YOUR neg_log_likelihood against the same formula written from the state it returns
    set_all_seeds(7)
    fe = Field(16, 3, zero_head=False).double(); de = AugmentedDynamics(fe, "hutchinson", probe_seed=3)
    nll_e, s0_e = neg_log_likelihood(de, two_moons(32, 4).double(), rtol=1e-8, atol=1e-10, adjoint=False)
    ref_e = 0.5 * (s0_e[:, :2] ** 2).sum(1) + LOG2PI + s0_e[:, 2]
    ps = list(fe.parameters())
    assert torch.allclose(nll_e, ref_e, atol=1e-12), "neg_log_likelihood differs from 0.5|x_0|^2 + log 2 pi + delta_logp(0)"
    g1 = torch.autograd.grad(nll_e.mean(), ps, retain_graph=True); g2 = torch.autograd.grad(ref_e.mean(), ps)
    assert all(torch.allclose(a, b, atol=1e-10) for a, b in zip(g1, g2)), "the gradient of neg_log_likelihood is not that of the formula: is something detached (a numpy call, .detach(), .item())?"
    print("(e) the gradient of your neg_log_likelihood matches the formula written from the state it returns.")
finally:
    torch.set_default_dtype(torch.float32)
print("Checkpoint 1.4 passed: the stated magnitude is a relative error of 1e-6; the last column is a dynamics whose")
print("vector-Jacobian product was built without create_graph.")

The last column of the table belongs to a class named `BrokenDynamics`, defined in the code cell before the checkpoint. Its
right-hand side is yours with the vector-Jacobian product built differently. Read its last column against the other two.

## Part 2 — Train under the frozen budget

The configuration is `cfg`, read from the YAML file. Nothing in this part is a TODO. The cell below defines
the training function with its learning-rate schedule, and three more given functions (the pictures and the tolerance sweep). Read the
training function once: adjoint mode, the U0.T2 defaults, a probe drawn once per minibatch solve, and the
instrumentation from iteration 0 (the step time, the number of calls of the field in the primal solve and
in the adjoint solve, the loss). **The names `nfe_fwd` and `nfe_bwd` are historical.** They count the calls
of the field in the primal (likelihood) solve and in the adjoint (sensitivity) solve. They do not name a
direction in time. A vector-Jacobian product made inside one call is not another call.

The training cell sets `DATASET = "two-moons"`, and the cells of Parts 3 and 4 set it again. Parts 2, 3 and 4 run on that dataset;
Part 5 repeats the same cells for the checkerboard, so that the outputs of both stay visible in the notebook.
Each training run is saved in `runs/`. **If the kernel restarts, run Parts 0 and 1 again, then the next two cells**: the first
defines the functions and the second reloads the saved runs, so that Parts 3 and 4 can go on without a new training.

In [ ]:
def warmup_cosine(step: int, total: int, warmup: int) -> float:
    """Multiplier on the base learning rate: the U0.T2 schedule, stepped per iteration."""
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * p))


def train_run(cfg: dict, dataset: str, *, seed: int | None = None, regularize: bool = False,
              iters: int | None = None, lr: float | None = None, verbose: bool = True):
    """Train under a budget config, instrumented from iteration 0. Returns (field, log).

    Adjoint mode, Hutchinson divergence with a probe fixed per solve, the U0.T2 defaults.
    Per iteration: the step time (solve + backward + update), the NFE of the primal solve,
    the NFE of the adjoint solve, the loss. Nothing is logged inside the step.
    """
    seed = cfg["seed"] if seed is None else seed
    iters = cfg["train"]["iters"] if iters is None else iters
    lr = cfg["opt"]["lr"] if lr is None else lr
    set_all_seeds(seed)
    train_x = GENERATORS[dataset](cfg["data"]["n_train"], cfg["data"]["seed_train"])
    field = Field(cfg["model"]["hidden"], cfg["model"]["depth"], cfg["model"]["zero_init_head"])
    w = (cfg["rnode"]["kinetic_weight"], cfg["rnode"]["jacobian_weight"]) if regularize else (0.0, 0.0)
    dyn = AugmentedDynamics(field, "hutchinson", regularize, w, probe_seed=seed + 1)
    opt = torch.optim.AdamW(field.parameters(), lr=lr, betas=tuple(cfg["opt"]["betas"]),
                            weight_decay=cfg["opt"]["weight_decay"])
    warm = max(1, round(cfg["sched"]["warmup_frac"] * iters))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: warmup_cosine(i, iters, warm))
    batch_gen = torch.Generator().manual_seed(seed)
    log = painplots.TrainLog()
    sol = cfg["solver"]
    t_start = time.perf_counter()
    assert (sol["method"], sol["adjoint"], cfg["model"]["activation"], cfg["train"]["log_every"]) == \
        ("dopri5", True, "tanh", painplots.LOG_EVERY), \
        "this code fixes the solver method, the adjoint mode, the activation and the logging cadence; the config records them"
    for it in range(1, iters + 1):
        idx = torch.randint(0, train_x.shape[0], (cfg["train"]["batch"],), generator=batch_gen)
        x1 = train_x[idx]
        t0 = time.perf_counter()
        nll, state0 = neg_log_likelihood(dyn, x1, rtol=sol["rtol"], atol=sol["atol"])
        loss = nll.mean()
        if regularize:
            loss = loss + w[0] * state0[:, 3].mean() + w[1] * state0[:, 4].mean()
        nfe_fwd = dyn.nfe
        opt.zero_grad(set_to_none=True)
        dyn.nfe = 0
        loss.backward()
        nfe_bwd = dyn.nfe
        opt.step()
        sched.step()
        step_s = time.perf_counter() - t0
        if not math.isfinite(loss.item()):
            raise FloatingPointError(f"loss is {loss.item()} at iteration {it}: use the fallback learning rate")
        log.add(step_s, nfe_fwd, nfe_bwd, nll.detach().mean().item())   # the NLL, so regularized runs compare
        if verbose and it % 500 == 0:
            print(f"  [{dataset} seed {seed}] it {it:5d}  nll {statistics.fmean(log.loss[-100:]):.3f}  "
                  f"nfe {statistics.fmean(log.nfe_fwd[-100:]):.0f}/{statistics.fmean(log.nfe_bwd[-100:]):.0f}  "
                  f"s/it {statistics.fmean(log.step_s[-100:]):.3f}  elapsed {(time.perf_counter() - t_start) / 60:.1f} min", flush=True)
    return field, log


@torch.no_grad()
def make_log_prob(field: nn.Module, rtol: float = 1e-5):
    def log_prob(pts: torch.Tensor) -> torch.Tensor:
        dyn = AugmentedDynamics(field, "exact")
        dyn.before_solve(pts)
        s0 = solve_augmented(dyn, pts, rtol=rtol, atol=rtol * 1e-2, adjoint=False)
        return -0.5 * (s0[:, :2] ** 2).sum(1) - LOG2PI - s0[:, 2]
    return log_prob


@torch.no_grad()
def make_generator(field: nn.Module):
    def gen(x0: torch.Tensor) -> torch.Tensor:
        f = lambda t, x: field(t, x)    # noqa: E731
        return odeint(f, x0, x0.new_tensor([0.0, 1.0]), rtol=1e-5, atol=1e-7, method="dopri5")[-1]
    return gen


def tolerance_rows(field: nn.Module, dataset: str, cfg: dict) -> list[dict]:
    x_test = GENERATORS[dataset](cfg["data"]["n_test"], cfg["data"]["seed_test"])
    grid = cfg["tolerance_grid"]
    rows = []
    for rtol in grid["rtol"]:
        r = evaluate_nll(field, x_test, rtol=rtol, atol=rtol * grid["atol_factor"])
        rows.append({"rtol": rtol, "atol": rtol * grid["atol_factor"], "nfe_eval": r["nfe"],
                     "nll_test": r["nll"], "wallclock_eval_s": r["wall_s"]})
    return rows

In [ ]:
# Reload the runs saved in runs/ (only needed after a restart of the kernel; it changes nothing otherwise).
for ds in ("two-moons", "checkerboard"):
    f = ROOT / "runs" / f"u2l2-{ds}.pt"
    if ds not in RESULTS and f.exists():
        r = torch.load(f, map_location="cpu")
        fld = Field(r["cfg"]["model"]["hidden"], r["cfg"]["model"]["depth"]); fld.load_state_dict(r["state_dict"]); fld.eval()
        lg = None
        if r["log"] is not None:
            lg = painplots.TrainLog(); lg.step_s, lg.nfe_fwd, lg.nfe_bwd, lg.loss = (r["log"][k] for k in ("step_s", "nfe_fwd", "nfe_bwd", "loss"))
        RESULTS[ds] = {"field": fld, "log": lg, "cfg": r["cfg"], "tier": r["tier"], "pair_id": r["pair_id"],
                       "device_kind": r["device_kind"], "device": r["device"], "machine_id": r["machine_id"],
                       "device_name": r["device_name"], "hardware_id": r.get("hardware_id", painplots.hardware_id())}
        print("reloaded the saved run of", ds)

In [ ]:
DATASET = "two-moons"
# Run the training (or load the reference run, tier 3). The run takes minutes; it needs nothing from you.
if TIER == 3:
    ck = torch.load(ROOT / "common" / "weights" / f"u2l2-{DATASET}.pt", map_location="cpu")
    field = Field(cfg["model"]["hidden"], cfg["model"]["depth"]); field.load_state_dict(ck["state_dict"]); field.eval()
    log = None
    print(f"tier 3: loaded the reference checkpoint for {DATASET}")
else:
    print(f"training {DATASET} (tier {TIER}, budget {cfg['budget_version']})")
    t0 = time.perf_counter()
    field, log = train_run(cfg, DATASET)
    print(f"{DATASET}: {len(log)} iterations in {time.perf_counter() - t0:.0f} s (tier {TIER}, budget {cfg['budget_version']})")
# the run keeps its own settings, so that Part 7 labels it correctly whatever TIER is set to later
RESULTS[DATASET] = {"field": field, "log": log, "cfg": copy.deepcopy(cfg), "tier": TIER, "pair_id": PAIR_ID,
                    "device_kind": DEVICE_KIND, "device": DEVICE, "machine_id": painplots.machine_id(),
                    "hardware_id": painplots.hardware_id(), "device_name": painplots.device_name(DEVICE)}
shutil.rmtree(EVIDENCE / DATASET, ignore_errors=True)    # a new run never inherits an evidence file of an earlier one
shutil.rmtree(ROOT / "mlflow_export" / DATASET, ignore_errors=True)    # ... or the export of an earlier one
# ... and it is saved, so that a restart of the kernel does not lose it (the second code cell of Part 2 reloads it)
(ROOT / "runs").mkdir(exist_ok=True)
torch.save({"state_dict": field.state_dict(), "cfg": RESULTS[DATASET]["cfg"], "tier": TIER, "pair_id": PAIR_ID,
            "device_kind": DEVICE_KIND, "device": DEVICE, "machine_id": RESULTS[DATASET]["machine_id"],
            "device_name": RESULTS[DATASET]["device_name"], "hardware_id": RESULTS[DATASET]["hardware_id"],
            "log": None if log is None else {"step_s": log.step_s, "nfe_fwd": log.nfe_fwd, "nfe_bwd": log.nfe_bwd, "loss": log.loss}},
           ROOT / "runs" / f"u2l2-{DATASET}.pt")

In [ ]:
# The learned density and the warped grid, with the toy2d helpers (the pictures of U1.L1).
DATASET = "two-moons"
field, log = RESULTS[DATASET]["field"], RESULTS[DATASET]["log"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
if log is not None:
    axes[0].plot([statistics.fmean(log.loss[i:i + 10]) for i in range(0, len(log), 10)], color="#003366")
    axes[0].set_xlabel("window of 10 iterations"); axes[0].set_ylabel("training NLL")
data = GENERATORS[DATASET](4096, 5)
axes[1].scatter(data[:, 0], data[:, 1], s=2, color="#6b7280", alpha=0.5); axes[1].set_xlim(*EXTENT); axes[1].set_ylim(*EXTENT)
density_heatmap(make_log_prob(field), n=100, ax=axes[2])
warped_grid(make_generator(field), ax=axes[3])
for a, tt in zip(axes, ("loss", "data", "learned density", "warped grid")):
    a.set_title(tt)
plt.tight_layout(); plt.show()

In [ ]:
# Checkpoint 2: the numbers you should see (a reference interval, not a threshold; it comes from a study of 5 seeds)
DATASET = "two-moons"
log, cfg_run, tier_run = RESULTS[DATASET]["log"], RESULTS[DATASET]["cfg"], RESULTS[DATASET]["tier"]
if log is None:
    print("Checkpoint 2: tier 3, no training run to check.")
else:
    n = len(log); k = 300 if n >= 1200 else n // 4
    first, last = statistics.fmean(log.loss[:k]), statistics.fmean(log.loss[-k:])
    assert all(math.isfinite(v) for v in log.loss), "non-finite loss: use TIER = 1"
    print(f"training NLL, first {k} iterations: {first:.2f}   last {k}: {last:.2f}")
    REF = {"two-moons": (2.05, 2.24),
           "checkerboard": (3.24, 3.37)}[DATASET]
    if cfg_run["budget_version"] == "U2L2-B1" and tier_run == 0:
        print(f"reference interval for the last {k}: {REF[0]:.2f} to {REF[1]:.2f}  ->", "inside" if REF[0] <= last <= REF[1] else "outside (a finite run outside the interval is still valid evidence)")
    assert last < first - 0.3, "the loss did not decrease: run Checkpoints 1.1 to 1.4 again before the rescue ladder"
    print("Checkpoint 2 passed: the loss is finite and it decreased.")

## Part 3 — The mandatory measurements

Two calls of `painplots`, no implementation of yours. The functions
write the CSV files of the contract and a PNG each. **The CSV schema is the contract; the PNG is a
courtesy.** Wall-clock is in seconds. NFE is the number of calls of the field.

In [ ]:
DATASET = "two-moons"
rs = RESULTS[DATASET]; log = rs["log"]; ev_dir = EVIDENCE / DATASET
if rs["tier"] == 3:
    shutil.rmtree(ev_dir, ignore_errors=True); shutil.copytree(REFERENCE / "pain_evidence" / DATASET, ev_dir)
    print(f"tier 3: copied the reference evidence for {DATASET} (evidence_origin=reference_pair)")
else:
    shutil.rmtree(ev_dir, ignore_errors=True)   # a new run never inherits a file of an earlier one
    rs.pop("rows", None)                        # ... and its tolerance sweep (Part 4) must be run again
    painplots.pain_wallclock(log, ev_dir)       # 1. per-iteration wall-clock over training
    painplots.pain_nfe_growth(log, ev_dir)      # 2. NFE of the primal and of the adjoint solve against iteration
for name in ("pain_wallclock", "pain_nfe_growth"):
    display(Image(str(ev_dir / f"{name}.png"), width=480))
print(open(ev_dir / "pain_wallclock.csv").read().splitlines()[:3])
print(open(ev_dir / "pain_nfe_growth.csv").read().splitlines()[:3])

**Prompt 3 (PS2.5 d), two-moons.** In two to four sentences, in your own words: describe how the NFE columns
change over the training, describe how the wall-clock column changes, and relate the two. Use course notation where
notation appears.

*Your answer:*

## Part 4 — Tolerance sensitivity

The sweep runs on the trained model with the **exact divergence**. This isolates the error of the ODE
solver from the Monte Carlo variance of the Hutchinson estimate. The solver is the training solver
(dopri5). The grid of the budget is `rtol` in $\{10^{-2}, \dots, 10^{-6}\}$ with `atol = 0.01 rtol`.
The test set is one batch of 2048 points; the NFE of an adaptive solve is a
property of the batch.

In [ ]:
DATASET = "two-moons"
rs = RESULTS[DATASET]; ev_dir = EVIDENCE / DATASET
rows = tolerance_rows(rs["field"], DATASET, rs["cfg"])
if rs["tier"] == 3:
    print("tier 3: the reference pain_tolerance files stay in the evidence directory (their wall-clock belongs to the reference computer); this sweep is printed only.")
else:
    painplots.pain_tolerance(rows, ev_dir)
ref_nll = rows[-1]["nll_test"]
print(f"{'rtol':>8} {'atol':>9} {'NFE':>5} {'test NLL':>10} {'|NLL - NLL(tightest)|':>22} {'seconds':>8}")
for r in rows:
    print(f"{r['rtol']:>8.0e} {r['atol']:>9.0e} {r['nfe_eval']:>5d} {r['nll_test']:>10.4f} {abs(r['nll_test'] - ref_nll):>22.4f} {r['wallclock_eval_s']:>8.1f}")
print("the picture below is the file of the evidence directory" + (" (the REFERENCE sweep, tier 3)" if rs["tier"] == 3 else ""))
display(Image(str(ev_dir / "pain_tolerance.png"), width=480))
rs["rows"] = rows; rs["rows_hardware"] = painplots.hardware_id()

**Prompt 4 (PS2.5 d), two-moons.** At which tolerance of the grid, if any, does the test NLL move by more than
$\varepsilon_{\mathrm{NLL}} = 0.01$ from its value at the tightest tolerance? In two to four sentences: what does
this say about the accuracy of the solver for *this* trained field? Do not infer that the same threshold holds
for another model or another dataset.

*Your answer:*

## Part 5 — The checkerboard

The cells below repeat the cells of Parts 2, 3 and 4 for the checkerboard. The cells differ from those above only in the dataset
they set, `DATASET = "checkerboard"`. The budget is the same for both datasets, unless you used a rescue tier for one of them. The evidence of
two-moons stays on disk in its own directory, and its outputs stay in the cells above. Before the training cell, look at `TIER` in the settings cell:
it applies to the run that follows.

In [ ]:
DATASET = "checkerboard"
# Run the training (or load the reference run, tier 3). The run takes minutes; it needs nothing from you.
if TIER == 3:
    ck = torch.load(ROOT / "common" / "weights" / f"u2l2-{DATASET}.pt", map_location="cpu")
    field = Field(cfg["model"]["hidden"], cfg["model"]["depth"]); field.load_state_dict(ck["state_dict"]); field.eval()
    log = None
    print(f"tier 3: loaded the reference checkpoint for {DATASET}")
else:
    print(f"training {DATASET} (tier {TIER}, budget {cfg['budget_version']})")
    t0 = time.perf_counter()
    field, log = train_run(cfg, DATASET)
    print(f"{DATASET}: {len(log)} iterations in {time.perf_counter() - t0:.0f} s (tier {TIER}, budget {cfg['budget_version']})")
# the run keeps its own settings, so that Part 7 labels it correctly whatever TIER is set to later
RESULTS[DATASET] = {"field": field, "log": log, "cfg": copy.deepcopy(cfg), "tier": TIER, "pair_id": PAIR_ID,
                    "device_kind": DEVICE_KIND, "device": DEVICE, "machine_id": painplots.machine_id(),
                    "hardware_id": painplots.hardware_id(), "device_name": painplots.device_name(DEVICE)}
shutil.rmtree(EVIDENCE / DATASET, ignore_errors=True)    # a new run never inherits an evidence file of an earlier one
shutil.rmtree(ROOT / "mlflow_export" / DATASET, ignore_errors=True)    # ... or the export of an earlier one
# ... and it is saved, so that a restart of the kernel does not lose it (the second code cell of Part 2 reloads it)
(ROOT / "runs").mkdir(exist_ok=True)
torch.save({"state_dict": field.state_dict(), "cfg": RESULTS[DATASET]["cfg"], "tier": TIER, "pair_id": PAIR_ID,
            "device_kind": DEVICE_KIND, "device": DEVICE, "machine_id": RESULTS[DATASET]["machine_id"],
            "device_name": RESULTS[DATASET]["device_name"], "hardware_id": RESULTS[DATASET]["hardware_id"],
            "log": None if log is None else {"step_s": log.step_s, "nfe_fwd": log.nfe_fwd, "nfe_bwd": log.nfe_bwd, "loss": log.loss}},
           ROOT / "runs" / f"u2l2-{DATASET}.pt")

In [ ]:
# The learned density and the warped grid, with the toy2d helpers (the pictures of U1.L1).
DATASET = "checkerboard"
field, log = RESULTS[DATASET]["field"], RESULTS[DATASET]["log"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
if log is not None:
    axes[0].plot([statistics.fmean(log.loss[i:i + 10]) for i in range(0, len(log), 10)], color="#003366")
    axes[0].set_xlabel("window of 10 iterations"); axes[0].set_ylabel("training NLL")
data = GENERATORS[DATASET](4096, 5)
axes[1].scatter(data[:, 0], data[:, 1], s=2, color="#6b7280", alpha=0.5); axes[1].set_xlim(*EXTENT); axes[1].set_ylim(*EXTENT)
density_heatmap(make_log_prob(field), n=100, ax=axes[2])
warped_grid(make_generator(field), ax=axes[3])
for a, tt in zip(axes, ("loss", "data", "learned density", "warped grid")):
    a.set_title(tt)
plt.tight_layout(); plt.show()

In [ ]:
# Checkpoint 2: the numbers you should see (a reference interval, not a threshold; it comes from a study of 5 seeds)
DATASET = "checkerboard"
log, cfg_run, tier_run = RESULTS[DATASET]["log"], RESULTS[DATASET]["cfg"], RESULTS[DATASET]["tier"]
if log is None:
    print("Checkpoint 2: tier 3, no training run to check.")
else:
    n = len(log); k = 300 if n >= 1200 else n // 4
    first, last = statistics.fmean(log.loss[:k]), statistics.fmean(log.loss[-k:])
    assert all(math.isfinite(v) for v in log.loss), "non-finite loss: use TIER = 1"
    print(f"training NLL, first {k} iterations: {first:.2f}   last {k}: {last:.2f}")
    REF = {"two-moons": (2.05, 2.24),
           "checkerboard": (3.24, 3.37)}[DATASET]
    if cfg_run["budget_version"] == "U2L2-B1" and tier_run == 0:
        print(f"reference interval for the last {k}: {REF[0]:.2f} to {REF[1]:.2f}  ->", "inside" if REF[0] <= last <= REF[1] else "outside (a finite run outside the interval is still valid evidence)")
    assert last < first - 0.3, "the loss did not decrease: run Checkpoints 1.1 to 1.4 again before the rescue ladder"
    print("Checkpoint 2 passed: the loss is finite and it decreased.")

In [ ]:
DATASET = "checkerboard"
rs = RESULTS[DATASET]; log = rs["log"]; ev_dir = EVIDENCE / DATASET
if rs["tier"] == 3:
    shutil.rmtree(ev_dir, ignore_errors=True); shutil.copytree(REFERENCE / "pain_evidence" / DATASET, ev_dir)
    print(f"tier 3: copied the reference evidence for {DATASET} (evidence_origin=reference_pair)")
else:
    shutil.rmtree(ev_dir, ignore_errors=True)   # a new run never inherits a file of an earlier one
    rs.pop("rows", None)                        # ... and its tolerance sweep (Part 4) must be run again
    painplots.pain_wallclock(log, ev_dir)       # 1. per-iteration wall-clock over training
    painplots.pain_nfe_growth(log, ev_dir)      # 2. NFE of the primal and of the adjoint solve against iteration
for name in ("pain_wallclock", "pain_nfe_growth"):
    display(Image(str(ev_dir / f"{name}.png"), width=480))
print(open(ev_dir / "pain_wallclock.csv").read().splitlines()[:3])
print(open(ev_dir / "pain_nfe_growth.csv").read().splitlines()[:3])

In [ ]:
DATASET = "checkerboard"
rs = RESULTS[DATASET]; ev_dir = EVIDENCE / DATASET
rows = tolerance_rows(rs["field"], DATASET, rs["cfg"])
if rs["tier"] == 3:
    print("tier 3: the reference pain_tolerance files stay in the evidence directory (their wall-clock belongs to the reference computer); this sweep is printed only.")
else:
    painplots.pain_tolerance(rows, ev_dir)
ref_nll = rows[-1]["nll_test"]
print(f"{'rtol':>8} {'atol':>9} {'NFE':>5} {'test NLL':>10} {'|NLL - NLL(tightest)|':>22} {'seconds':>8}")
for r in rows:
    print(f"{r['rtol']:>8.0e} {r['atol']:>9.0e} {r['nfe_eval']:>5d} {r['nll_test']:>10.4f} {abs(r['nll_test'] - ref_nll):>22.4f} {r['wallclock_eval_s']:>8.1f}")
print("the picture below is the file of the evidence directory" + (" (the REFERENCE sweep, tier 3)" if rs["tier"] == 3 else ""))
display(Image(str(ev_dir / "pain_tolerance.png"), width=480))
rs["rows"] = rows; rs["rows_hardware"] = painplots.hardware_id()

**Prompt 5 (PS2.5 d), observation only.** Compare the learned density of the checkerboard, and the
boundaries of its eight modes, with what you remember of your U1.L1 fit. Answer in two to four sentences; one of
them cites the `topology-obstruction-argument` of U1.T2 as the statement you compare your observation with.
Describe what you see; do not look for a cause.

*Your answer:*

## Part 6 — Optional: the kinetic and Jacobian penalties

Both penalties are already wired behind `regularize=True` in `train_run`; they are given, not a TODO.
Run two-moons twice at half the iterations, without and with the penalties, and overlay the NFE curves.
This part is optional and ungraded. **Its results never enter the evidence directories.** The runs are
logged to MLflow with the tag `optional=rnode`.

The training solves of the two runs do not have the same state: three columns without the penalties and five
with them, and the step-size controller measures its error over all the columns. The NFE of the two training
solves is therefore not a like-for-like count. The cell also evaluates both final fields under one and the same
plain solve (exact divergence, `rtol` $10^{-6}$), and prints the NFE of that solve. The overlay is saved as
`rnode_overlay.png` in the repository root, beside `pain_evidence/`, and logged as an artifact of the two optional MLflow runs; the export
of Part 7 holds the two mandatory runs only.

In [ ]:
RUN_OPTIONAL = False        # set to True to run Part 6 (two runs of half the iterations each)
if RUN_OPTIONAL:
    cfg6 = yaml.safe_load((ROOT / "configs" / "u2l2-budget.yaml").read_text())       # Part 6 always uses the frozen budget
    half = cfg6["train"]["iters"] // 2
    x_test = GENERATORS["two-moons"](cfg6["data"]["n_test"], cfg6["data"]["seed_test"])
    runs = {}
    for label, reg in (("without penalties", False), ("with penalties", True)):
        fld, lg = train_run(cfg6, "two-moons", iters=half, regularize=reg, verbose=False)
        runs[label] = (fld, lg, reg, evaluate_nll(fld.eval(), x_test, rtol=1e-6, atol=1e-8))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for label, (fld, lg, reg, ev) in runs.items():
        for ax, series in zip(axes, (lg.nfe_fwd, lg.nfe_bwd)):
            ax.plot([statistics.fmean(series[i:i + 10]) for i in range(0, len(series), 10)], label=label)
    axes[0].set_title("primal solve"); axes[1].set_title("adjoint solve"); axes[0].legend()
    for ax in axes: ax.set_xlabel("window of 10 iterations"); ax.set_ylabel("NFE per solve")
    plt.tight_layout(); plt.savefig(ROOT / "rnode_overlay.png", dpi=150); plt.show()
    for label, (fld, lg, reg, ev) in runs.items():
        print(f"{label}: test NLL {ev['nll']:.3f}; NFE of one evaluation solve (exact divergence, rtol 1e-6): {ev['nfe']}")
        painplots.log_run(None, run_name=f"u2l2-rnode-{label.replace(' ', '-')}", log=lg, tracking_uri="sqlite:///mlflow.db",
                          tags={"session": "U2.L2", "dataset": "two-moons", "optional": "rnode", "regularize": str(reg).lower(),
                                "hardware": DEVICE_KIND}, params={"iters": half, "seed": cfg["seed"]},
                          extra_artifacts=[ROOT / "rnode_overlay.png"])

**Optional prompt.** Did the NFE of the training solves change, and did the NFE of the common evaluation solve
change? In which direction, and by how much? Describe what the curves and the printed numbers show for these two runs.

*Your answer:*

## Part 7 — Package and epilogue

The cell below writes `meta.json` for each dataset, runs the validator of PS2.5 (b), logs each run to
MLflow and exports a portable copy of it. **The validator must print PASS for both directories.** The
export (`mlflow_export/<dataset>/`) is what you submit: it holds the parameters, the tags and the metrics
of the run, and no path of your computer. At tier 3 the reference `meta.json` stays in the directory, and the export
holds no metrics, because that run has no training log.

In [ ]:
ok = True
for ds in ("two-moons", "checkerboard"):
    if ds not in RESULTS:
        print(f"{ds}: has no training run in this notebook. Run its training cell (Part {'2' if ds == 'two-moons' else '5'}); after a restart of the kernel run the two code cells of Part 2 that define the functions and reload the saved runs."); ok = False; continue
    d = EVIDENCE / ds
    shutil.rmtree(ROOT / "mlflow_export" / ds, ignore_errors=True)    # a failed check never leaves an older export behind
    rs = RESULTS[ds]
    cfg_r, tier_r = rs["cfg"], rs["tier"]           # the settings of THIS run, whatever TIER is set to now
    if "rows" not in rs:
        print(f"{ds}: the evidence cell (Part 3) and the tolerance cell (Part 4) have not both run for this run since the kernel started. Run them (for the checkerboard: the evidence and tolerance cells of Part 5)."); ok = False; continue
    if tier_r != 3 and rs["rows_hardware"] != rs["hardware_id"]:
        print(f"{ds}: the tolerance sweep and the training ran on two computers, so its wall-clock column would carry the wrong machine. Run the training and Part 4 on one computer. (A virtual machine can change its hardware address at a restart: the same remedy holds.)"); ok = False; continue
    if tier_r != 3:
        painplots.pain_wallclock(rs["log"], d)      # the two files are rewritten from THIS run's log: no stale file can be labelled as yours
        painplots.pain_nfe_growth(rs["log"], d)
    elif (d / "meta.json").exists() and painplots.read_evidence(d)["meta"]["evidence_origin"] != "reference_pair":
        print(f"{ds}: this run is tier 3 but the directory holds your own evidence. Run the Part 3 cell of this dataset again."); ok = False; continue
    if tier_r != 3:
        meta = painplots.make_meta(ds, cfg_r["seed"], {**cfg_r, "dataset": ds}, pair_id=rs["pair_id"],
                                   budget_version=cfg_r["budget_version"], evidence_origin="student",
                                   divergence_mode="hutchinson_rademacher", device=rs["device"])
        meta["machine_id"], meta["device_name"] = rs["machine_id"], rs["device_name"]      # measured on THAT computer, at training time
        painplots.write_meta(meta, d)
        (d / "resolved_config.yaml").write_text(yaml.safe_dump({**cfg_r, "dataset": ds}, sort_keys=True))
    valid = painplots.print_validation(d, expected_session="U2.L2", label=Path("pain_evidence") / ds)
    ok &= valid
    if not valid:
        print(f"  {ds}: fix the directory above, then run this cell again"); continue
    meta = painplots.read_evidence(d)["meta"]
    run_id = painplots.log_run(
        d, run_name=f"u2l2-{ds}", log=rs["log"], tracking_uri="sqlite:///mlflow.db",
        tags={"session": "U2.L2", "dataset": ds, "hardware": rs["device_kind"], "machine_id": meta["machine_id"],
              "pair_id": meta["pair_id"], "evidence_origin": meta["evidence_origin"], "rescue": str(tier_r > 0).lower(),
              "tier": str(tier_r)},
        params={"budget_version": meta["budget_version"], "seed": cfg_r["seed"], "lr": cfg_r["opt"]["lr"],
                "iters": cfg_r["train"]["iters"], "hidden": cfg_r["model"]["hidden"], "config_hash": meta["config_hash"]})
    painplots.export_run(run_id, ROOT / "mlflow_export" / ds, tracking_uri="sqlite:///mlflow.db")
    print(f"  MLflow run {run_id[:8]} exported to mlflow_export/{ds}/")
own_ids = {RESULTS[ds]["pair_id"] for ds in RESULTS if RESULTS[ds]["tier"] != 3}
pair_file = EVIDENCE / "pair.json"
if own_ids and own_ids != {json.loads(pair_file.read_text())["pair_id"] if pair_file.exists() else None}:
    print("note: the pair_id of your runs is not the one in pain_evidence/pair.json, because the machine id changed (the host name is part of it) or the file was recreated. U3.L1 pairs runs by this id: say so in the thread (#lab).")
print("\nREADY TO SUBMIT" if ok else "\nNOT READY: see the lines above")

### PS2.5 checklist

*Deliverable: this lab, packaged. Grading follows the course norm — correctness and reproducibility, not
sample quality.*

- (a) Completed notebook, all TODOs, all checkpoint cells executed with outputs visible.
- (b) `pain_evidence/two-moons/` and `pain_evidence/checkerboard/`, complete per the layout of
  `common/painplots.py`, with a well-formed `meta.json` (the validator cell prints PASS).
- (c) The gradient-check table (Part 1, Checkpoint 1.4) within the stated magnitudes.
- (d) Written answers to the three prompt cells (Parts 3, 4 and 5), two to four sentences each, in course
  notation where notation appears.
- (e) `mlflow_export/` for the two mandatory runs. Runs of the rescue ladder are admissible if tagged
  `rescue=true`. There is no penalty; the tag is for the instructor's statistics.
- (f) *Optional, ungraded:* the Part 6 overlay plot `rnode_overlay.png`; the optional runs carry the tag `optional=rnode` in MLflow.

Send these items as one ZIP file to the instructor's institutional e-mail. Do not add `mlflow.db`, `runs/` or
the rest of your clone. Use the subject `PS2.5 <your name>`. The pinned message of the thread states the address
and the due date. Keep the folders in your clone.

You now hold two plots that measure the cost of learning by simulation. Keep them where you can find them for U3.L1 — and reread [the question on the board](https://adricortes.github.io/flow-matching-lectures/U2/U2-T2-notes.html#sec-question).